<a href="https://colab.research.google.com/github/elenaajayi/spec-gap-activation-probe/blob/main/notebooks/01_sanity_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [0]:
# Artifact directory setup
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_ARTIFACT_ROOT = Path("/content/drive/MyDrive/spec-gap-activation-probe/artifacts")
except Exception:
    DEFAULT_ARTIFACT_ROOT = Path.cwd() / "artifacts"

ARTIFACT_ROOT = Path(os.environ.get("SPEC_GAP_ARTIFACT_ROOT", DEFAULT_ARTIFACT_ROOT))
ARTIFACT_DIR = ARTIFACT_ROOT / "01_sanity_check"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Artifact root: {ARTIFACT_ROOT}")
print(f"This notebook writes to: {ARTIFACT_DIR}")


# Week 1 Sanity Check: Residual Stream Extraction + Geometry of Truth

Validates the extraction pipeline on Llama 3 8B Instruct by replicating
Marks & Tegmark (2023) true/false linear separability.

Please use a H100 GPU for this or your session will crash A LOT!

**Requirements:** HuggingFace account with access to meta-llama/Meta-Llama-3-8B-Instruct

In [0]:
# 1. Install dependencies
!pip install -q transformer-lens torch einops jaxtyping scikit-learn

In [0]:
# 2. HuggingFace login
# Uses HF_TOKEN_SANITY env var. Set it in Colab via:
#   Secrets (key icon in left sidebar) -> add HF_TOKEN_SANITY
# Or uncomment the manual line below.
import os
from huggingface_hub import login
from google.colab import userdata

token = userdata.get("HF_TOKEN_SANITY") # Use userdata.get for Colab Secrets
if token:
    login(token=token)
    print("Logged in via HF_TOKEN_SANITY")
else:
    login()  # fallback: manual prompt
    # raise EnvironmentError("Set HF_TOKEN_SANITY in Colab Secrets or environment")

In [0]:
# 3. Extraction module

import torch
from transformer_lens import HookedTransformer

DEFAULT_LAYERS = (16, 20, 24)
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
HIDDEN_DIM = 4096


def load_model(model_name=MODEL_NAME, device="cuda", dtype=torch.float16):
    model = HookedTransformer.from_pretrained(
        model_name,
        device=device,
        dtype=dtype,
    )
    model.eval()
    return model


def _get_cache_filter(layer_to_extract):
    # Filter for a single layer
    def name_filter(name):
        if not name.endswith("hook_resid_post"):
            return False
        parts = name.split(".")
        layer_idx = int(parts[1])
        return layer_idx == layer_to_extract
    return name_filter


def extract_residual_stream(model, prompts, layers=DEFAULT_LAYERS,
                            token_position="last", batch_size=1):
    results = {layer: [] for layer in layers}

    # Process layer by layer to reduce peak GPU memory usage
    for layer in layers:
        name_filter = _get_cache_filter(layer)
        layer_activations = []

        for i in range(0, len(prompts), batch_size):
            batch = prompts[i : i + batch_size]
            tokens = model.to_tokens(batch, prepend_bos=True)

            with torch.no_grad():
                _, cache = model.run_with_cache(
                    tokens,
                    names_filter=name_filter,
                )

            hook_name = f"blocks.{layer}.hook_resid_post"
            activations = cache[hook_name]

            if token_position == "last":
                act = activations[:, -1, :]
            elif token_position == "all":
                act = activations
            elif isinstance(token_position, int):
                act = activations[:, token_position, :]
            else:
                raise ValueError(f"Unknown token_position: {token_position}")

            layer_activations.append(act.cpu())

            del cache
            torch.cuda.empty_cache()

        results[layer] = torch.cat(layer_activations, dim=0)

    return results

In [0]:
# 4. True/false statement sets (30 each)

TRUE_STATEMENTS = [
    "The Earth orbits the Sun.",
    "Water boils at 100 degrees Celsius at sea level.",
    "The speed of light is approximately 300,000 kilometers per second.",
    "Humans have 23 pairs of chromosomes.",
    "The chemical formula for water is H2O.",
    "The Great Wall of China is visible from space with aid.",
    "DNA stands for deoxyribonucleic acid.",
    "The Pacific Ocean is the largest ocean on Earth.",
    "Diamonds are made of carbon.",
    "The Moon causes tides on Earth.",
    "Shakespeare wrote Hamlet.",
    "Oxygen is necessary for human respiration.",
    "The Nile is the longest river in Africa.",
    "Iron is attracted to magnets.",
    "Photosynthesis converts sunlight into chemical energy.",
    "The human heart has four chambers.",
    "Jupiter is the largest planet in our solar system.",
    "Sound travels faster in water than in air.",
    "The boiling point of nitrogen is below zero degrees Celsius.",
    "Antibiotics are used to treat bacterial infections.",
    "The Amazon rainforest produces a significant portion of the world's oxygen.",
    "Gold is a chemical element with the symbol Au.",
    "The Sahara is the largest hot desert in the world.",
    "Insulin regulates blood sugar levels.",
    "Light travels faster than sound.",
    "The human body contains more bacteria than human cells.",
    "Mount Everest is the tallest mountain above sea level.",
    "Electrons have a negative charge.",
    "The liver is the largest internal organ in the human body.",
    "Gravity pulls objects toward the center of the Earth.",
]

FALSE_STATEMENTS = [
    "The Earth is the largest planet in the solar system.",
    "Water freezes at 50 degrees Celsius.",
    "The speed of light is approximately 300 kilometers per second.",
    "Humans have 46 pairs of chromosomes.",
    "The chemical formula for water is CO2.",
    "The Great Wall of China was built in the 20th century.",
    "DNA stands for dioxyribonucleic acid.",
    "The Atlantic Ocean is the largest ocean on Earth.",
    "Diamonds are made of iron.",
    "The Sun causes tides on Earth.",
    "Shakespeare wrote The Odyssey.",
    "Nitrogen is necessary for human respiration.",
    "The Amazon is the longest river in Africa.",
    "Wood is attracted to magnets.",
    "Photosynthesis converts moonlight into chemical energy.",
    "The human heart has two chambers.",
    "Saturn is the largest planet in our solar system.",
    "Sound travels faster in a vacuum than in air.",
    "The boiling point of nitrogen is above 100 degrees Celsius.",
    "Antibiotics are used to treat viral infections.",
    "The Sahara desert produces a significant portion of the world's oxygen.",
    "Gold is a chemical element with the symbol Fe.",
    "Antarctica is the largest hot desert in the world.",
    "Adrenaline regulates blood sugar levels.",
    "Sound travels faster than light.",
    "The human body contains no bacteria.",
    "Mount Kilimanjaro is the tallest mountain above sea level.",
    "Electrons have a positive charge.",
    "The spleen is the largest internal organ in the human body.",
    "Gravity pushes objects away from the center of the Earth.",
]

In [0]:
# 5. Load model
print(f"Loading {MODEL_NAME}...")
model = load_model(device="cuda")
print(f"Loaded. {model.cfg.n_layers} layers, hidden dim {model.cfg.d_model}")

In [0]:
# 6. Extract activations
all_prompts = TRUE_STATEMENTS + FALSE_STATEMENTS
print(f"Extracting activations for {len(all_prompts)} statements at layers {DEFAULT_LAYERS}...")

activations = extract_residual_stream(model, all_prompts, batch_size=1)

for layer, acts in activations.items():
    print(f"  Layer {layer}: {acts.shape}")

# Free GPU memory
del model
torch.cuda.empty_cache()
print("Model unloaded.")

In [0]:
# 7. Train probes and compute AUROC
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

labels = np.array([1] * len(TRUE_STATEMENTS) + [0] * len(FALSE_STATEMENTS))

results = {}
for layer, acts in activations.items():
    X = acts.numpy().astype(np.float32)
    X_train, X_test, y_train, y_test = train_test_split(
        X, labels, test_size=0.3, random_state=42, stratify=labels,
    )

    probe = LogisticRegression(max_iter=1000, C=1.0)
    probe.fit(X_train, y_train)

    y_prob = probe.predict_proba(X_test)[:, 1]
    auroc = roc_auc_score(y_test, y_prob)
    accuracy = probe.score(X_test, y_test)

    mean_true = X[labels == 1].mean(axis=0)
    mean_false = X[labels == 0].mean(axis=0)
    diff_norm = np.linalg.norm(mean_true - mean_false)

    results[layer] = {
        "auroc": auroc,
        "accuracy": accuracy,
        "diff_means_norm": float(diff_norm),
        "n_train": len(y_train),
        "n_test": len(y_test),
    }

    status = "PASS" if auroc > 0.8 else "CHECK"
    print(f"Layer {layer}: AUROC={auroc:.3f}  Acc={accuracy:.3f}  ||diff||={diff_norm:.1f}  [{status}]")

In [0]:
# 8. Visualize
import matplotlib.pyplot as plt

layers_sorted = sorted(results.keys())
aurocs = [results[l]["auroc"] for l in layers_sorted]
accs = [results[l]["accuracy"] for l in layers_sorted]

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.bar([str(l) for l in layers_sorted], aurocs, color=["#2196F3", "#4CAF50", "#FF9800"])
ax.axhline(y=0.8, color="red", linestyle="--", label="Threshold (0.8)")
ax.axhline(y=0.5, color="gray", linestyle=":", label="Chance")
ax.set_xlabel("Layer")
ax.set_ylabel("AUROC")
ax.set_title("True/False Separability by Layer (Llama 3.1 8B Instruct)")
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

In [0]:
# 7b. Re-run with 5-fold CV + PCA (fixes small-N noise from cell 7)
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

labels = np.array([1] * len(TRUE_STATEMENTS) + [0] * len(FALSE_STATEMENTS))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results_cv = {}
for layer, acts in activations.items():
    X = acts.numpy().astype(np.float32)

    fold_aurocs = []
    fold_accs = []

    for train_idx, test_idx in cv.split(X, labels):
        pipe = Pipeline([
            ("pca", PCA(n_components=48)), # Changed from 50 to 48
            ("probe", LogisticRegression(max_iter=1000, C=1.0)),
        ])
        pipe.fit(X[train_idx], labels[train_idx])

        y_prob = pipe.predict_proba(X[test_idx])[:, 1]
        fold_aurocs.append(roc_auc_score(labels[test_idx], y_prob))
        fold_accs.append(pipe.score(X[test_idx], labels[test_idx]))

    mean_auroc = np.mean(fold_aurocs)
    std_auroc = np.std(fold_aurocs)
    mean_acc = np.mean(fold_accs)

    mean_true = X[labels == 1].mean(axis=0)
    mean_false = X[labels == 0].mean(axis=0)
    diff_norm = np.linalg.norm(mean_true - mean_false)

    results_cv[layer] = {
        "auroc_mean": float(mean_auroc),
        "auroc_std": float(std_auroc),
        "auroc_per_fold": [float(a) for a in fold_aurocs],
        "accuracy_mean": float(mean_acc),
        "diff_means_norm": float(diff_norm),
    }

    status = "PASS" if mean_auroc > 0.8 else "CHECK"
    print(f"Layer {layer}: AUROC={mean_auroc:.3f} \u00b1 {std_auroc:.3f}  Acc={mean_acc:.3f}  [{status}]")

In [0]:
# 8b. Compare single-split vs CV
import matplotlib.pyplot as plt

layers_sorted = sorted(results_cv.keys())
aurocs_cv = [results_cv[l]["auroc_mean"] for l in layers_sorted]
stds_cv = [results_cv[l]["auroc_std"] for l in layers_sorted]
aurocs_orig = [results[l]["auroc"] for l in layers_sorted]

x = np.arange(len(layers_sorted))
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.bar(x - 0.15, aurocs_orig, 0.3, label="Single split (cell 7)", color="#BBDEFB")
ax.bar(x + 0.15, aurocs_cv, 0.3, yerr=stds_cv, capsize=5, label="5-fold CV + PCA (cell 7b)", color="#2196F3")
ax.axhline(y=0.8, color="red", linestyle="--", label="Threshold (0.8)")
ax.axhline(y=0.5, color="gray", linestyle=":", label="Chance")
ax.set_xticks(x)
ax.set_xticklabels([str(l) for l in layers_sorted])
ax.set_xlabel("Layer")
ax.set_ylabel("AUROC")
ax.set_title("True/False Separability: Single Split vs 5-Fold CV + PCA")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [0]:
# 9b. Save CV results
import json
from datetime import datetime

output_cv = {
    "experiment": "sanity_check_geometry_of_truth_cv",
    "model": MODEL_NAME,
    "method": "5-fold stratified CV, PCA(50) + LogisticRegression",
    "date": datetime.now().isoformat(),
    "n_true": len(TRUE_STATEMENTS),
    "n_false": len(FALSE_STATEMENTS),
    "token_position": "last",
    "layers": {str(k): v for k, v in results_cv.items()},
}

with open(ARTIFACT_DIR / "sanity_check_results_cv.json", "w") as f:
    json.dump(output_cv, f, indent=2)

print(json.dumps(output_cv, indent=2))
print(f"Saved CV results to {ARTIFACT_DIR / 'sanity_check_results_cv.json'}")

In [0]:
# 9. Save results
import json
from datetime import datetime

output = {
    "experiment": "sanity_check_geometry_of_truth",
    "model": MODEL_NAME,
    "date": datetime.now().isoformat(),
    "n_true": len(TRUE_STATEMENTS),
    "n_false": len(FALSE_STATEMENTS),
    "token_position": "last",
    "layers": {str(k): v for k, v in results.items()},
}

with open(ARTIFACT_DIR / "sanity_check_results.json", "w") as f:
    json.dump(output, f, indent=2)

print(json.dumps(output, indent=2))
print(f"Saved sanity check results to {ARTIFACT_DIR / 'sanity_check_results.json'}")

layer_20 = results.get(20, {})
if layer_20.get("auroc", 0) > 0.8:
    print("\nSanity check PASSED: layer 20 AUROC > 0.8")
else:
    print(f"\nSanity check NEEDS REVIEW: layer 20 AUROC = {layer_20.get('auroc', 'N/A')}")